In [1]:
import torch

import arc_compressor
import initializers
import preprocessing

TASK_ID = "272f95fa"
POLICY = "relaxed"

task = preprocessing.preprocess_tasks(
    "training",
    [TASK_ID],
    multitensor_constraints=POLICY,
)[0]

initializer = initializers.Initializer(
    task.multitensor_system,
    arc_compressor.ARCCompressor(task).channel_dim_fn,
)

multiposteriors = initializer.initialize_multiposterior(
    arc_compressor.ARCCompressor.decoding_dim
)

axis_names = ("E", "C", "D", "H", "W")

print("Task:", TASK_ID)
print("Policy:", POLICY)
print("Dimension lengths:")
print({
    "E": task.n_examples,
    "C": task.n_colors,
    "D": 8,
    "H": task.n_x,
    "W": task.n_y,
})
print()

for dims in task.multitensor_system:
    posterior_mean, local_capacity_adjustment = multiposteriors[dims]

    active_axes = tuple(
        name for name, present in zip(axis_names, dims)
        if present
    )

    print(
        f"{str(active_axes):25s}",
        f"dims={tuple(dims)}",
        f"mean={tuple(posterior_mean.shape)}",
        f"local_capacity={tuple(local_capacity_adjustment.shape)}",
    )

Task: 272f95fa
Policy: relaxed
Dimension lengths:
{'E': 3, 'C': 6, 'D': 8, 'H': 18, 'W': 19}

('E',)                    dims=(1, 0, 0, 0, 0) mean=(3, 4) local_capacity=(3, 4)
('C',)                    dims=(0, 1, 0, 0, 0) mean=(6, 4) local_capacity=(6, 4)
('E', 'C')                dims=(1, 1, 0, 0, 0) mean=(3, 6, 4) local_capacity=(3, 6, 4)
('D',)                    dims=(0, 0, 1, 0, 0) mean=(8, 4) local_capacity=(8, 4)
('E', 'D')                dims=(1, 0, 1, 0, 0) mean=(3, 8, 4) local_capacity=(3, 8, 4)
('C', 'D')                dims=(0, 1, 1, 0, 0) mean=(6, 8, 4) local_capacity=(6, 8, 4)
('E', 'C', 'D')           dims=(1, 1, 1, 0, 0) mean=(3, 6, 8, 4) local_capacity=(3, 6, 8, 4)
('H',)                    dims=(0, 0, 0, 1, 0) mean=(18, 4) local_capacity=(18, 4)
('E', 'H')                dims=(1, 0, 0, 1, 0) mean=(3, 18, 4) local_capacity=(3, 18, 4)
('C', 'H')                dims=(0, 1, 0, 1, 0) mean=(6, 18, 4) local_capacity=(6, 18, 4)
('E', 'C', 'H')           dims=(1, 1, 0, 1, 0) m

In [2]:
import arc_compressor
import initializers
import preprocessing

TASK_ID = "272f95fa"
POLICY = "relaxed"

task = preprocessing.preprocess_tasks(
    "training",
    [TASK_ID],
    multitensor_constraints=POLICY,
)[0]

initializer = initializers.Initializer(
    task.multitensor_system,
    channel_dim_fn=lambda dims: 16 if dims[2] == 0 else 8,
)

multiposteriors = initializer.initialize_multiposterior(
    decoding_dim=4
)

axis_names = ("E", "C", "D", "H", "W")

for dims in task.multitensor_system:
    mean, local_capacity_adjustment = multiposteriors[dims]
    name = ",".join(
        axis for axis, enabled in zip(axis_names, dims)
        if enabled
    )

    print(
        f"({name:9s}) "
        f"mean={tuple(mean.shape)}, "
        f"local={tuple(local_capacity_adjustment.shape)}"
    )

(E        ) mean=(3, 4), local=(3, 4)
(C        ) mean=(6, 4), local=(6, 4)
(E,C      ) mean=(3, 6, 4), local=(3, 6, 4)
(D        ) mean=(8, 4), local=(8, 4)
(E,D      ) mean=(3, 8, 4), local=(3, 8, 4)
(C,D      ) mean=(6, 8, 4), local=(6, 8, 4)
(E,C,D    ) mean=(3, 6, 8, 4), local=(3, 6, 8, 4)
(H        ) mean=(18, 4), local=(18, 4)
(E,H      ) mean=(3, 18, 4), local=(3, 18, 4)
(C,H      ) mean=(6, 18, 4), local=(6, 18, 4)
(E,C,H    ) mean=(3, 6, 18, 4), local=(3, 6, 18, 4)
(D,H      ) mean=(8, 18, 4), local=(8, 18, 4)
(E,D,H    ) mean=(3, 8, 18, 4), local=(3, 8, 18, 4)
(C,D,H    ) mean=(6, 8, 18, 4), local=(6, 8, 18, 4)
(E,C,D,H  ) mean=(3, 6, 8, 18, 4), local=(3, 6, 8, 18, 4)
(W        ) mean=(19, 4), local=(19, 4)
(E,W      ) mean=(3, 19, 4), local=(3, 19, 4)
(C,W      ) mean=(6, 19, 4), local=(6, 19, 4)
(E,C,W    ) mean=(3, 6, 19, 4), local=(3, 6, 19, 4)
(D,W      ) mean=(8, 19, 4), local=(8, 19, 4)
(E,D,W    ) mean=(3, 8, 19, 4), local=(3, 8, 19, 4)
(C,D,W    ) mean=(6, 8, 19, 4)

In [3]:
dims = [0, 1, 0, 1, 1]

mean, local_capacity_adjustment = multiposteriors[dims]

print("Semantic dimensions:", "(C,H,W)")
print("Mean shape:", mean.shape)
print("Local capacity shape:", local_capacity_adjustment.shape)

Semantic dimensions: (C,H,W)
Mean shape: torch.Size([6, 18, 19, 4])
Local capacity shape: torch.Size([6, 18, 19, 4])
